# Options

Every bar takes `options=psnek.Options(...)`, a set of presentation settings that can be shared by any number of bars. This notebook demonstrates each one:

| Option           | Default      | Effect                                                                  | Section                                                           |
| ---------------- | ------------ | ----------------------------------------------------------------------- | ----------------------------------------------------------------- |
| `unit`           | `"it"`       | Label for counts and rates                                              | [Units and Scale](#Units-and-Scale)                               |
| `scale`          | `"none"`     | `"si"` (1000) or `"iec"` (1024) number prefixes                         | [Units and Scale](#Units-and-Scale)                               |
| `height`         | `1`          | Rows the bar spans                                                      | [Height](#Height)                                                 |
| `gradient`       | `True`       | Color the bar with its gradient, your own hex colors, or the text color | [Gradient](#Gradient)                                             |
| `leave`          | `None`       | Keep the final line once it closes (`None`: top-level bars only)        | [Leave](#Leave)                                                   |
| `disable`        | `False`      | Draw nothing, while counting and text still work                        | [Disable](#Disable)                                               |
| `stream`         | `sys.stderr` | Where the bar is drawn                                                  | [Stream](#Stream)                                                 |
| `panel_lines`    | `8`          | Max lines the panel shows                                               | [The Panel](#The-Panel)                                           |
| `panel_justify`  | `"left"`     | Alignment of table columns in the panel                                 | [The Panel](#The-Panel)                                           |
| `panel_position` | `"below"`    | Panel `"below"` or `"above"` the bars                                   | [The Panel](#The-Panel)                                           |
| `panel_clear`    | `False`      | Remove the panel's content when the bars finish                         | [The Panel](#The-Panel)                                           |
| `log_time`       | `False`      | Start each `bar.log()` message with the time                            | [Panel notebook](02_panel_tables.ipynb#Log-Levels-and-Timestamps) |
| `capture`        | `True`       | In a terminal, route `print()` and logging into the panel               | [Terminal Only](#Terminal-Only:-Capture-and-Notify)               |
| `notify`         | `True`       | In a terminal, allow desktop notifications about the bar                | [Terminal Only](#Terminal-Only:-Capture-and-Notify)               |

Panel and capture settings come from the first bar in a cell, so each panel example below runs in its own cell.


In [1]:
import fcntl
import io
import os
import pty
import struct
import subprocess
import sys
import termios
import textwrap
import time

import pyte

import psnek

psnek.__version__

'0.1.0'

## Units and Scale

`unit` names what is counted. `scale` sets the number prefixes: `"none"` shows plain counts, `"si"` divides by 1000 (`k`, `M`, `G`), and `"iec"` by 1024 (`Ki`, `Mi`, `Gi`). The same 3.5 GB transfer with each:


In [2]:
SIZE, CHUNK = 3_500_000_000, 70_000_000

for scale in ("none", "si", "iec"):
    options = psnek.Options(unit="B", scale=scale)
    with psnek.Bar(total=SIZE, desc=f"upload, scale={scale!r}", options=options) as bar:
        for _ in range(SIZE // CHUNK):
            time.sleep(0.01)
            bar.update(CHUNK)

# upload, scale='none' ############  100.0%      3.50G/3.50G         0:00   5.83GB/s  done
# upload, scale='si'   ############  100.0%    3.50GB/3.50GB         0:00   5.79GB/s  done
# upload, scale='iec'  ############  100.0%  3.26GiB/3.26GiB         0:00  5.45GiB/s  done

## Height

`height=n` draws the bar over `n` rows. The same run at one, two, and three rows (the [tour](01_tour.ipynb#Taller-Bars) has more):


In [3]:
DESC = "fine-tune llama-3-8b on support tickets, epoch 1/1"

for height in (1, 2, 3):
    options = psnek.Options(height=height, unit="step")
    with psnek.Bar(total=120, desc=DESC, options=options) as bar:
        for step in range(120):
            time.sleep(0.008)
            loss = 0.4 + 2.1 * 0.97**step
            bar.status({"train loss": loss, "lr": 3e-4, "rows": height})
            bar.update()

# fine-tune llama-3-8b on suppor... ##########  100.0%    120/120         0:01   102step/s  done
# fine-tune llama-3-8b on support   ##########  100.0%    120/120         0:01   102step/s  done
  tickets, epoch 1/1                ##########  train loss=0.4560  lr=0.0003000  rows=2
# fine-tune llama-3-8b on support   ##########  100.0%    120/120         0:01   103step/s  done
  tickets, epoch 1/1                ##########  train loss=0.4560  rows=3
                                    ##########  lr=0.0003000

## Gradient

`gradient=False` draws running and finished bars in the text color instead of the violet-to-green gradient, and turns off the shimmer that glides along them. A bar that stalls, gets stuck, or fails still changes color, so trouble stands out. The same download with and without the gradient:


In [4]:
for gradient in (True, False):
    options = psnek.Options(unit="B", scale="iec", gradient=gradient)
    desc = f"download, gradient={gradient}"
    with psnek.Bar(total=SIZE, desc=desc, options=options) as bar:
        for _ in range(SIZE // CHUNK):
            time.sleep(0.01)
            bar.update(CHUNK)

# download, gradient=True  ##########  100.0%  3.26GiB/3.26GiB         0:00  5.42GiB/s  done
# download, gradient=False ##########  100.0%  3.26GiB/3.26GiB         0:00  5.45GiB/s  done

A list or tuple of `"#RRGGBB"` colors replaces the gradient, spread evenly from the left of the bar to the right, with the shimmer still gliding along it:


In [5]:
options = psnek.Options(unit="B", scale="iec", gradient=["#A855F7", "#0EA5E9"])
with psnek.Bar(total=SIZE, desc="download, purple to blue", options=options) as bar:
    for _ in range(SIZE // CHUNK):
        time.sleep(0.01)
        bar.update(CHUNK)

# download, purple to blue ##########  100.0%  3.26GiB/3.26GiB         0:00  5.32GiB/s  done

## Leave

`leave` decides whether a bar's final line stays once it closes. By default, only top-level bars are kept:


In [6]:
def train(epochs: psnek.Options, batches: psnek.Options) -> None:
    """Run three epochs of 20 batches, each epoch with its own nested bar."""
    for epoch in psnek.track(range(3), "epochs", options=epochs):
        name = f"epoch {epoch + 1} batches"
        for _ in psnek.track(range(20), name, options=batches):
            time.sleep(0.01)


train(psnek.Options(), psnek.Options())

# epochs            #####################  100.0%        3/3         0:00    4.1it/s  done

`leave=True` on the nested bars keeps each of them too:


In [7]:
train(psnek.Options(), psnek.Options(leave=True))

# epoch 1 batches   #####################  100.0%      20/20         0:00   82.1it/s  done
# epoch 2 batches   #####################  100.0%      20/20         0:00   81.9it/s  done
# epoch 3 batches   #####################  100.0%      20/20         0:00   83.1it/s  done
# epochs            #####################  100.0%        3/3         0:00    4.1it/s  done

`leave=False` on the top-level bar removes it as well:


In [8]:
train(psnek.Options(leave=False), psnek.Options())
print("the cell output holds no bars")

the cell output holds no bars


## Disable

`disable=True` draws no bar, which is useful for a `verbose` flag. Counting, outcomes, `log()`, and `write()` still work.


In [9]:
verbose = False

options = psnek.Options(disable=not verbose)
with psnek.Bar(total=50, desc="warm cache", options=options) as bar:
    for _ in range(50):
        bar.update()
    bar.log("cache warmed")
print(f"n={bar.n}, outcome={bar.outcome}")

info  cache warmed


n=50, outcome=done


## Stream

`stream` sets where the bar is drawn. A stream that isn't a terminal or notebook, such as a file or a buffer, gets plain status lines instead.


In [10]:
buffer = io.StringIO()
options = psnek.Options(stream=buffer, unit="row")
with psnek.Bar(total=200, desc="export", options=options) as bar:
    for _ in range(200):
        time.sleep(0.005)
        bar.update()
    bar.status({"rows per file": 5000})
print(buffer.getvalue(), end="")

export: 0.0% 0/200 0:00<--:-- --row/s
export: 100.0% 200/200 0:01 done 164row/s rows per file=5000


## The Panel

This loop shows a table of 12 crawler workers in the panel. With the defaults, the panel shows at most 8 lines, so the table ends with a count of the rows that don't fit (the [panel notebook](02_panel_tables.ipynb) has more):


In [11]:
def crawl(options: psnek.Options, steps: int = 30) -> None:
    """Crawl for ``steps`` steps, showing a table of worker states in the panel."""
    with psnek.Bar(total=steps, desc="crawl", options=options) as bar:
        for step in range(steps):
            time.sleep(0.03)
            bar.panel(
                [
                    {
                        "worker": worker,
                        "state": "busy" if (step + worker) % 3 else "idle",
                        "pages": step * (worker + 1) * 7,
                        "MiB": round(step * (worker + 1) * 0.37, 1),
                    }
                    for worker in range(12)
                ]
            )
            bar.update()


crawl(psnek.Options())

# crawl #################################  100.0%      30/30         0:01   29.6it/s  done
| worker  state  pages  MiB
| 0       busy   203    10.70
| 1       idle   406    21.50
| 2       busy   609    32.20
| 3       busy   812    42.90
| 4       idle   1015   53.60
| 5       busy   1218   64.40
| ... 6 more rows

`panel_lines=4` keeps the header and two rows:


In [12]:
crawl(psnek.Options(panel_lines=4))

# crawl #################################  100.0%      30/30         0:01   29.7it/s  done
| worker  state  pages  MiB
| 0       busy   203    10.70
| 1       idle   406    21.50
| ... 10 more rows

`panel_justify="right"` aligns every column to the right:


In [13]:
crawl(psnek.Options(panel_justify="right"))

# crawl #################################  100.0%      30/30         0:01   29.4it/s  done
| worker  state  pages    MiB
|      0   busy    203  10.70
|      1   idle    406  21.50
|      2   busy    609  32.20
|      3   busy    812  42.90
|      4   idle   1015  53.60
|      5   busy   1218  64.40
| ... 6 more rows

`panel_position="above"` draws the panel above the bars:


In [14]:
crawl(psnek.Options(panel_position="above"))

| worker  state  pages  MiB
| 0       busy   203    10.70
| 1       idle   406    21.50
| 2       busy   609    32.20
| 3       busy   812    42.90
| 4       idle   1015   53.60
| 5       busy   1218   64.40
| ... 6 more rows
# crawl #################################  100.0%      30/30         0:01   28.9it/s  done

`panel_clear=True` removes the panel's content once the bars finish:


In [15]:
crawl(psnek.Options(panel_clear=True))

# crawl #################################  100.0%      30/30         0:01   29.0it/s  done

## Terminal Only: Capture and Notify

These two options only apply in a terminal. To demonstrate them here, `terminal()` runs a script in a pseudo-terminal (POSIX only) and replays its output through [pyte](https://github.com/selectel/pyte), a terminal emulator, to show the final screen.


In [16]:
def terminal(
    script: str, columns: int = 90, rows: int = 8, **env: str
) -> tuple[str, bytes]:
    """Run ``script`` on a pseudo-terminal ``columns`` by ``rows`` cells.

    Args:
        script: Python source, dedented before it runs.
        columns: Terminal width.
        rows: Terminal height.
        **env: Extra environment variables for the child.

    Returns:
        The final screen as text, and the raw bytes the child wrote.
    """
    leader, follower = pty.openpty()
    size = struct.pack("HHHH", rows, columns, 0, 0)
    fcntl.ioctl(follower, termios.TIOCSWINSZ, size)
    child = subprocess.Popen(  # noqa: S603 - runs only this notebook's scripts
        [sys.executable, "-c", textwrap.dedent(script)],
        stdin=subprocess.DEVNULL,
        stdout=follower,
        stderr=follower,
        env={**os.environ, "PSNEK_MODE": "live", **env},
    )
    os.close(follower)
    chunks = []
    while True:
        try:
            chunk = os.read(leader, 65536)
        except OSError:  # Linux reports EIO once the child closes the terminal
            break
        if not chunk:
            break
        chunks.append(chunk)
    child.wait()
    os.close(leader)
    written = b"".join(chunks)
    screen = pyte.Screen(columns, rows)
    pyte.ByteStream(screen).feed(written)
    return "\n".join(line.rstrip() for line in screen.display).rstrip(), written

`capture=True` (the default) routes `print()` and logging output into the panel. With `capture=False`, each print lands on top of the live bar, and the next redraw overwrites it.


In [17]:
TRAIN = """
    import time
    import psnek

    options = psnek.Options(capture={capture})
    with psnek.Bar(total=30, desc="train", options=options) as bar:
        for step in range(30):
            time.sleep(0.05)
            if step % 10 == 9:
                print(f"checkpoint saved at step {{step + 1}}")
            bar.update()
"""

for capture in (True, False):
    screen, _ = terminal(TRAIN.format(capture=capture))
    print(f"capture={capture}:\n{screen}\n")

capture=True:
⣿ train ⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿  100.0%      30/30         0:01   18.4it/s  done
checkpoint saved at step 10
checkpoint saved at step 20
checkpoint saved at step 30



capture=False:
⡆ train ⣿⣿⣿⣿⣿⣿⡏⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀⣀   30.0%       9/30   0:00<~0:01   17.2it/s
⣠heckpoint⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⣀⣀⣀⣀⣀⣀⣀⣀   63.3%      19       0:01<0:00   18.0
⠸heckpoint saved⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡄   96.7%      29                   18.3
⣿ train ⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿  100.0%      30/30         0:01   18.6it/s  done



`notify=True` (the default) lets psnek send a desktop notification when the bar gets stuck, fails, or finishes a long run. A pseudo-terminal isn't a known terminal and this run is short, so `PSNEK_NOTIFY` and `PSNEK_NOTIFY_AFTER` force notifications on here. With `notify=False`, none is sent.


In [18]:
EXPORT = """
    import time
    import psnek

    options = psnek.Options(notify={notify})
    with psnek.Bar(total=5, desc="nightly export", options=options) as bar:
        for _ in range(5):
            time.sleep(0.05)
            bar.update()
"""

for notify in (True, False):
    _, written = terminal(
        EXPORT.format(notify=notify), PSNEK_NOTIFY="osc777", PSNEK_NOTIFY_AFTER="0"
    )
    sent = [
        part.split(b"\x1b\\")[0].decode().split(";", 1)
        for part in written.split(b"\x1b]777;notify;")[1:]
    ]
    shown = [f"title={title!r}, body={body!r}" for title, body in sent]
    print(f"notify={notify}:", *shown or ["nothing sent"])

notify=True: title='nightly export done', body='100.0% 5/5 0:00 done 18.6it/s'


notify=False: nothing sent
